# 02. Baseline Inference: Hugging Face vs llama.cpp
## LLM Inference Benchmark & Optimization Study

### Objectives
1. Measure steady-state token generation on a baseline CPU execution path.
2. Compare the runtime characteristics of Hugging Face Transformers vs llama.cpp.
3. Record Time to First Token (TTFT), Inter-Token Latency (ITL), and memory footprint.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.inference_bench.engines.llama_cpp_engine import LlamaCppEngine
from src.inference_bench.metrics import MetricsCalculator
import time

model_path = "../models/qwen2.5-0.5b-instruct-q4_k_m.gguf"
engine = LlamaCppEngine(model_path, config={"n_threads": 2, "n_ctx": 512})
engine.load_model()
print("Model loaded successfully.")


In [ ]:
prompt = "Explain why the sky appears blue in one sentence."
t0 = time.perf_counter()
output_text, token_timestamps, p_tokens, g_tokens = engine.generate(prompt, max_new_tokens=24)
t_end = time.perf_counter()

metrics = MetricsCalculator.compute(t0, token_timestamps, t_end, p_tokens)
print(f"Generated text: {output_text}")
print(f"TTFT: {metrics['ttft_ms']} ms")
print(f"Median ITL: {metrics['median_itl_ms']} ms")
print(f"Throughput: {metrics['output_tokens_per_second']} tokens/sec")
engine.unload_model()
